# &nbsp;&nbsp;&nbsp;<span style="color: #f83707ff;">此处有一个好消息！</span>
#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;你已经阅读本文档至第三章了，我们将迎来第一次软件层面上的升级：对老朋友Numpy数组说再见吧，Cupy才是更棒的！开个玩笑，实际上是因为本章之后的神经网络规模只会越来越大，Numpy的CPU计算已经不堪重负，所以我们改用语法几乎一样 对GPU特别优化的Cupy库。不过这对Macbook用户来说可能不是喜讯，由于Cupy对CUDA生态的依赖，Mac对Cupy的支持非常有限。一种妥协是将本文档的Cupy部分替换为Numpy，不过那样会花上几倍的时间运行模型。请各位读者前往NVIDIA官网安装CUDA并配置环境变量，详细参考教程。如果你还是很不舍得离别Numpy，别担心，我们还会使用到CPU的计算，因此他今后也不会完全缺席。
##### <span style="color: #b8cdcfff;">明明是我先来的。全连接也好，Adam也好，还是识别MNIST也好。为什么会变成这样呢？第一次有了喜欢的模型，有了能计算一辈子的模型。两件快乐的事重合在一起...但是，为什么会这样呢？...
</span>

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;为了让我们前几章的结果跟上软件的更新，我们在Layers_Optimizers_cp.py中重写了所有工具，后续我们会多多使用。

# &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1387abff;">Chapter 3: </span>卷积神经网络的构建

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;所谓卷积神经网络(Convolutional Neural Network)，是一种模拟生物视觉构建的神经网络模型。传统的全连接神经网络将图像展平一维化再做特征提取，如此必然会损失原图像在高维上的特征。因此，我们需要一种尽可能保留原图像特征并提取他们的神经网络。理论上，这种模型在图像识别上会更加出色。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;那么，我们如何做到这一点呢？答案是卷积层(Convolution Layer)。每个卷积层中都有多个卷积核，这些卷积核通过局部感受野(Local Receptive Field)与权值共享(Weight Sharing)的方式，既保留了图像的空间结构，又大大减少了模型参数数量。局部感受野意味着每个神经元只关注输入图像的一小部分区域，而权值共享则意味着同一卷积核在整个图像上滑动时使用相同的权重。这种设计使得卷积神经网络能够有效地捕捉图像中的局部特征，同时保持对整体结构的敏感性。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;我承认上这段写得完全不通俗。为了帮助读者理解卷积层的底层，首先需要理解卷积核具体的特征提取方式。例如对于一张(1,28,28)的单通道图像，设置一个(1,4,4)卷积核，步长为1。为了保证处理后图像尺寸不会剧烈缩小，我们在原图高与宽上填充1单位，即高与宽各增加2单位长度。那么输出宽度是(28-4+2)/1+1 = 27，输出高度则是一样的，所以输出尺寸是27*27 = 729。而卷积核扫描整个图像所需的步数就是结果的尺寸大小。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;实现CNN最大的难点，是多维张量的处理。在FCNN中，你可能已经默认数据集是三维的，形如(number,height,width)，然而对于实际图像，往往还要多出一维的通道，即(number,channel,height,width)。这是因为在RGB视角下，每张图片都可以分为三层甚至更多。对于每个通道，我们都专门在卷积核中设置一个对应的通道处理，这被称为单通道滤波器。一个比喻是，处理一辆车的图像，有一个通道负责观察轮子，一个负责观察车门，还有负责看车灯 引擎的等等。所以卷积核也可以写成(filter_number,channel,filter_height,filter_width)的形式。那么，既然已经有多通道了，为什么还要多个卷积核？另一个比喻是，外科医生可能很懂外科的每个细节，但是对汽车就不一定了解，所以我们还需要汽车专家。这可能可以解答你的疑惑。

# 卷积核处理结果尺寸计算公式

### 分别计算宽度和高度
$$
\text{输出宽度} = \frac{W_{\text{in}} - F_w + 2P_w}{S_w} + 1
$$

$$
\text{输出高度} = \frac{H_{\text{in}} - F_h + 2P_h}{S_h} + 1
$$

## 参数说明
- $W_{\text{in}}$, $H_{\text{in}}$: 输入特征的宽度和高度
- $F_w$, $F_h$: 卷积核的宽度和高度
- $P_w$, $P_h$: 宽度和高度方向的填充数
- $S_w$, $S_h$: 宽度和高度方向的步长

## 卷积神经网络核心公式

### 1. 卷积计算
$$
\text{Output}[i,j] = \sum_{m=0}^{FH}\sum_{n=0}^{FW} \text{Input}[i+m,j+n] \times \text{Kernel}[m,n] + b
$$

### 2. ReLU 激活函数
$$
\text{FeatureMap} = \max(0, \text{Output})
$$

### 3. 最大池化
$$
\text{Pooled}[i,j] = \max(\text{FeatureMap})
$$

### 4. 整体流程
输入 → 卷积 → ReLU → 池化 → 输出


#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;说了这么多，恐怕还是要动手实现才行吧。但请不要着急，我们先介绍一个非常重要的函数，im2col( Image to Column )，可以将卷积运算转换为矩阵运算。之所以需要这样做，是因为卷积运算不规则不连续，而矩阵乘法已经被现代计算机高度优化，非常适合GPU进行运算。im2col所做的是，将需要被卷积运算的数字展平为一维向量，再将这些向量组合成矩阵进行运算。卷积运算将卷积核各个位置对应的数字分别相乘( Hadamard积 )，这与向量乘法有天然的相似性。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;一个很好的理解是，一张图像是一个大立方体(C,H,W)，而一个卷积核是一个小立方体(C,FH,FW)，卷积核处理图像就是小立方体在大立方体内沿一个二维平面运动并扫描的过程。而如果把大立方体每一步被扫描的小立方体展平为行向量，再把这些行向量组装成大矩阵，同时卷积核展平为列向量，每个元素的展平顺序与大立方体中的小立方体被展平顺序相对应。那么，卷积的过程就可以直接理解为矩阵与向量的乘积。若有多个卷积核，则可以理解为矩阵与矩阵的乘积。最后输出时只需把结果矩阵拼回立方体。

In [1]:
import cupy as cp
def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    """
    An implementation of im2col (Image to Column) function.
    This function rearranges image blocks into columns.

    Parameters:
    - input_data: Input data of shape (N, C, H, W)
    - filter_h: Height of the filter
    - filter_w: Width of the filter
    - stride: Stride size
    - pad: Padding size

    Returns:
    - col: 2D array where each row is a flattened receptive field
    """
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h) // stride + 1
    out_w = (W + 2*pad - filter_w) // stride + 1

    img = cp.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant')#在高度和宽度维度上进行填充
    col = cp.zeros((N, C, filter_h, filter_w, out_h, out_w))

    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]#提取卷积核每个元素的感受野并存储到col数组中

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)#使用transpose将维度顺序从(N,C,fh,fw,oh,ow)改为(N,oh,ow,C,fh,fw)
                                                                    #使用reshape将结果转换为2D矩阵，形状为(N*oh*ow, C*fh*fw)
                                                                    #每行代表一个感受野的展平形式
                                                                    #行数为输出特征图的元素总数(N×oh×ow)
                                                                    #列数为每个感受野的元素总数(C×fh×fw)
    return col


#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;一个解释是，我们把所有被扫描的小立方体的集合称为输入窗口，reshape为( N * OH * OW，C* FH * FW )，将卷积核reshape为( FN，C * FH * FW )，两者( 转置 )的乘积就是( FN，N * OH * OW)。所以最后只需要将乘积矩阵reshape为( N，FN，OH，OW )即可。如果从这个逻辑出发，明白最后几行代码的动机就变得容易多了：首先，如何组装这个输入窗口？img的形状是( N，C，H_pad，W_pad )，一个img有N * OH * OW个需要被扫描的小立方体。我们考虑卷积核每个位置能够扫描到的对应小立方体的元素，这就是img[:, :, y:y_max:stride, x:x_max:stride]，形状是( N，C，OH，OW)，因此正适合存入col对应( :，:，y，x，:，: )的位置。最后这一步转换维度是最巧妙的，你可以理解为我们换了一种视角观察这个六维数组：原本我们考虑卷积核每个位置储存的被乘元素，变换后，我们考虑每个行向量对应一个被扫描的小立方体。这部分确实很烧脑，请读者多想想吧。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;我们刚刚提到，数据集矩阵与滤波器矩阵相乘得到输出矩阵，那么输出矩阵怎么转回一个高维张量？我们需要col2im。

In [2]:
def col2im(col, input_shape, filter_h, filter_w, stride=1, pad=0):
    """
    An implementation of col2im (Column to Image) function.
    This function rearranges columns back into image blocks.

    Parameters:
    - col: 2D array where each row is a flattened receptive field
    - input_shape: Shape of the original input data (N, C, H, W)
    - filter_h: Height of the filter
    - filter_w: Width of the filter
    - stride: Stride size
    - pad: Padding size

    Returns:
    - img: Reconstructed image data of shape (N, C, H, W)
    """
    N, C, H, W = input_shape
    out_h = (H + 2*pad - filter_h) // stride + 1
    out_w = (W + 2*pad - filter_w) // stride + 1
    col = col.reshape(N, out_h, out_w, C, filter_h, filter_w).transpose(0, 3, 4, 5, 1, 2)

    img = cp.zeros((N, C, H + 2*pad + stride - 1, W + 2*pad + stride - 1))
    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            img[:, :, y:y_max:stride, x:x_max:stride] += col[:, :, y, x, :, :]#将col中的数据累加回img的对应位置

    return img[:, :, pad:H + pad, pad:W + pad]

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;col2im与im2col大致原理是一样的。需要注意的是，最后将col中数据转回img的操作很有意思。由于我们的同一个像素可能出现在多个感受野中（重叠区域），所以在反向传播时需要将所有相关的梯度累加。你可以发现，im2col与col2im都是为卷积层的各个步骤特别准备的。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;卷积层的实现如下。

In [3]:
class Convolution:
    def __init__(self, W, b, stride=1, pad=0):
        self.W = W  # Filter weights
        self.b = b  # Biases
        self.stride = stride
        self.pad = pad
        self.col = None
        self.x = None
        # gradients
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape
        out_h = (H + 2*self.pad - FH) // self.stride + 1
        out_w = (W + 2*self.pad - FW) // self.stride + 1

        self.col = im2col(x, FH, FW, self.stride, self.pad)
        col_W = self.W.reshape(FN, -1).T  # Reshape filter weights (FN, C*FH*FW) to (C*FH*FW, FN)

        out = cp.dot(self.col, col_W) + self.b  # Matrix multiplication and add bias
        out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)  # Reshape and transpose to (N, FN, out_h, out_w)

        return out
    
    def backward(self, dout):
        FN, C, FH, FW = self.W.shape#( N, FN, out_h, out_w )
        dout_reshaped = dout.transpose(0, 2, 3, 1).reshape(-1, FN)  # reshape to ( N*out_h*out_w, FN )

        col_W = self.W.reshape(FN, -1).T #( C*FH*FW, FN ) to ( FN, C*FH*FW )

        dcol = cp.dot(dout_reshaped, col_W.T) # ( N*out_h*out_w, C*FH*FW ) × ( FN, C*FH*FW ).T 
        dW = cp.dot(self.col.T, dout_reshaped)# ( C*FH*FW, N*out_h*out_w ) × ( N*out_h*out_w, FN )
        dW = dW.transpose(1, 0).reshape(FN, C, FH, FW)

        db = cp.sum(dout_reshaped, axis=0)

        dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)

        # store gradients in layer attributes for later retrieval
        self.dW = dW
        self.db = db

        return dx

#### 我之前一直觉得Perplexity是个搜索引擎Agent，但是他给的反向传播解释说得还不错：实际上卷积层反向传播与Affine层区别并不大，核心公式是
$$
前向传播公式：

\text{out}_{\text{flat}} = \text{col} \cdot \text{col}_W + b
$$
$$
其中：
\text{col} = \text{im2col}(x, \text{FH}, \text{FW}, \text{stride}, \text{pad})\\
$$
$$
\text{col}_W = W.\text{reshape}(\text{FN}, -1)^T
$$
$$
反向传播 - 输入梯度推导：
\frac{\partial L}{\partial \text{col}} = \frac{\partial L}{\partial \text{out}_{\text{flat}}} \cdot \frac{\partial \text{out}_{\text{flat}}}{\partial \text{col}}
$$
$$
\frac{\partial \text{out}_{\text{flat}}}{\partial \text{col}} = \text{col}_W^T
$$
$$
\text{dcol} = \text{dout}_{\text{reshaped}} \cdot \text{col}_W^T\\
$$
$$
\frac{\partial L}{\partial x} = \text{col2im}(\text{dcol}, x.\text{shape}, \text{FH}, \text{FW}, \text{stride}, \text{pad})\\
$$
$$
反向传播 - 权重梯度推导：

\frac{\partial L}{\partial \text{col}_W} = \frac{\partial L}{\partial \text{out}_{\text{flat}}} \cdot \frac{\partial \text{out}_{\text{flat}}}{\partial \text{col}_W}\\
$$
$$
\frac{\partial \text{out}_{\text{flat}}}{\partial \text{col}_W} = \text{col}^T\\
$$
$$
\frac{\partial L}{\partial \text{col}_W} = \text{col}^T \cdot \text{dout}_{\text{reshaped}}\\
$$
$$
\text{dW} = \left( \text{col}^T \cdot \text{dout}_{\text{reshaped}} \right)^T.\text{reshape}(\text{FN}, C, \text{FH}, \text{FW})\\
$$
$$
反向传播 - 偏置梯度推导：

\frac{\partial L}{\partial b} = \sum_{i=1}^{N \cdot \text{out}_h \cdot \text{out}_w} \frac{\partial L}{\partial \text{out}_{\text{flat}}[i,:]}\\
$$
$$
\text{db} = \sum_{\text{axis}=0} \text{dout}_{\text{reshaped}}\\
$$
$$
完整的链式法则表达（卷积符号形式）：

\frac{\partial L}{\partial x} = \frac{\partial L}{\partial z} * \text{rot}_{180°}(W)\\
$$
$$
\frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} * x\\
$$
$$
\frac{\partial L}{\partial b} = \sum \frac{\partial L}{\partial z}\\
$$



#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;之所以说二者区别不大，是因为我们只是把梯度反复展开再拼回去而已，这个过程对真实的梯度不会有任何数值改变。可以看到卷积层的核心公式与线性层一摸一样，因此读者们也不要把这件事想得过于复杂。不过区别是，卷积层拥有权值共享，你可以理解为全连接层上一层的所有神经元与下一层所有神经元相连接，卷积层的每个神经元却只与某部分神经元相连接。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;另一个部分是池化层。所谓池化层，是对卷积层输出的特征图进行降维（下采样）的层​，压缩数据和参数的数量，同时保留最重要的特征信息。实际上池化层不是必需的，但绝大多数成功的CNN架构中都使用了它，原因是：首先，池化降低了计算量与内存消耗；其次，通过减少参数量，防止了过拟合；如果输入图像中的目标物体发生微小的平移（比如猫的位置向左移动了几个像素），池化后的输出特征基本保持不变，这被称为池化层的平移不变性；最后，池化层还为后方的卷积层扩大了感受野。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;池化层的操作和卷积层非常相似，也是用一个小窗口在特征图上滑动，但它没有需要学习的权重参数，仅仅是对窗口内结果取平均或取最大值。其中，最大池化是最常用有效的方法。如果我们只取窗口内最大的元素，那么其他元素的梯度就不必计算了，还带来了对数据的鲁棒性( 非最大元素变动不改变池化结果 )。

In [4]:
class Pooling:
    def __init__(self, pool_h, pool_w, stride=1, pad=0):
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad
        self.x = None
        self.arg_max = None

    def forward(self, x):
        N, C, H, W = x.shape
        out_h = (H - self.pool_h) // self.stride + 1
        out_w = (W - self.pool_w) // self.stride + 1

        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h * self.pool_w)

        arg_max = cp.argmax(col, axis=1)
        out = cp.max(col, axis=1)

        out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)

        self.x = x
        self.arg_max = arg_max

        return out

    def backward(self, dout):
        dout = dout.transpose(0, 2, 3, 1)
        pool_size = self.pool_h * self.pool_w
        dmax = cp.zeros((dout.size, pool_size))
        dmax[cp.arange(self.arg_max.size), self.arg_max.flatten()] = dout.flatten()
        dmax = dmax.reshape(dout.shape + (pool_size,))

        dcol = dmax.reshape(-1, pool_size)
        dx = col2im(dcol, self.x.shape, self.pool_h, self.pool_w, self.stride, self.pad)

        return dx

# Helper flatten layer to connect conv/pool to affine
class Flatten:
    def __init__(self):
        self.x_shape = None

    def forward(self, x):
        self.x_shape = x.shape
        return x.reshape(x.shape[0], -1)

    def backward(self, dout):
        return dout.reshape(self.x_shape)

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;我们还定义了一个Flatten层，这是因为我们需要将卷积层与池化层的最终结果输入线性层，这一步需要将特征图展平。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;各位读者，我们最终组装一个拥有三层卷积层与两层线性层的卷积神经网络。我们重写了BatchNorm的逻辑，原因是对线性层与卷积层需要不同的标准化方式。同时，考虑到笔者与各位读者阅读设备的性能，我们修改了predict函数，采用抽取批次预测的方式，否则非常容易出现OOM( Out of Memory )的情况。笔记本与家用台式机GPU显存有限，而模型参数往往非常庞大，因此这样的修改是必须的。

In [5]:
from Layers_Optimizers_cp import Affine, ReLU, SoftmaxWithLoss, Adam, BatchNorm, Dropout, RMSprop, to_one_hot
import cupy as cp
import math

class ConvNet:
    def __init__(self, input_dim=(1,28,28), hidden_size=100, output_size=10):
        """深层卷积神经网络，使用三层卷积层(32-64-128)和BatchNorm
        
        Parameters:
        - input_dim: (C, H, W)
        - hidden_size: 全连接层的隐藏单元数
        - output_size: 输出类别数
        """
        input_size = input_dim[1]
        filter_stride = 1
        filter_pad = 1
        
        # 三层卷积的参数设置
        conv_channels = [32, 64, 128]  # 逐层增加的卷积核数量
        
        # 计算每层卷积后的特征图大小
        current_size = input_size
        for _ in range(3):  # 3个卷积层，每个后面都跟着一个2x2池化
            current_size = (current_size + 2*filter_pad - 3)/filter_stride + 1  # conv
            current_size = current_size // 2  # pool
            
        # 最后一层池化后的特征数
        pool_output_size = int(conv_channels[2] * current_size * current_size)
        
        self.params = {}
        
        # 第一层卷积：输入通道->32
        fan_in = input_dim[0] * 3 * 3
        self.params['W1'] = cp.random.randn(conv_channels[0], input_dim[0], 3, 3).astype(cp.float32) * math.sqrt(2.0/fan_in)
        self.params['B1'] = cp.zeros(conv_channels[0], dtype=cp.float32)
        self.params['bn1_gamma'] = cp.ones(conv_channels[0], dtype=cp.float32)
        self.params['bn1_beta'] = cp.zeros(conv_channels[0], dtype=cp.float32)
        
        # 第二层卷积：32->64
        fan_in = conv_channels[0] * 3 * 3
        self.params['W2'] = cp.random.randn(conv_channels[1], conv_channels[0], 3, 3).astype(cp.float32) * math.sqrt(2.0/fan_in)
        self.params['B2'] = cp.zeros(conv_channels[1], dtype=cp.float32)
        self.params['bn2_gamma'] = cp.ones(conv_channels[1], dtype=cp.float32)
        self.params['bn2_beta'] = cp.zeros(conv_channels[1], dtype=cp.float32)
        
        # 第三层卷积：64->128
        fan_in = conv_channels[1] * 3 * 3
        self.params['W3'] = cp.random.randn(conv_channels[2], conv_channels[1], 3, 3).astype(cp.float32) * math.sqrt(2.0/fan_in)
        self.params['B3'] = cp.zeros(conv_channels[2], dtype=cp.float32)
        self.params['bn3_gamma'] = cp.ones(conv_channels[2], dtype=cp.float32)
        self.params['bn3_beta'] = cp.zeros(conv_channels[2], dtype=cp.float32)
        
        # 全连接层
        self.params['W4'] = cp.random.randn(pool_output_size, hidden_size).astype(cp.float32) * math.sqrt(2.0/pool_output_size)
        self.params['B4'] = cp.zeros(hidden_size, dtype=cp.float32)
        self.params['W5'] = cp.random.randn(hidden_size, output_size).astype(cp.float32) * math.sqrt(2.0/hidden_size)
        self.params['B5'] = cp.zeros(output_size, dtype=cp.float32)
        
        # 构建网络层
        self.layers = {}
        # Conv1 -> BN1 -> ReLU -> Pool
        self.layers['Conv1'] = Convolution(self.params['W1'], self.params['B1'], filter_stride, filter_pad)
        self.layers['BatchNorm1'] = BatchNorm(conv_channels[0], is_conv_layer=True)  # 标记为卷积层的BN
        self.layers['Relu1'] = ReLU()
        self.layers['Pool1'] = Pooling(pool_h=2, pool_w=2, stride=2)
        
        # Conv2 -> BN2 -> ReLU -> Pool
        self.layers['Conv2'] = Convolution(self.params['W2'], self.params['B2'], filter_stride, filter_pad)
        self.layers['BatchNorm2'] = BatchNorm(conv_channels[1], is_conv_layer=True)  # 标记为卷积层的BN
        self.layers['Relu2'] = ReLU()
        self.layers['Pool2'] = Pooling(pool_h=2, pool_w=2, stride=2)
        
        # Conv3 -> BN3 -> ReLU -> Pool
        self.layers['Conv3'] = Convolution(self.params['W3'], self.params['B3'], filter_stride, filter_pad)
        self.layers['BatchNorm3'] = BatchNorm(conv_channels[2], is_conv_layer=True)  # 标记为卷积层的BN
        self.layers['Relu3'] = ReLU()
        self.layers['Pool3'] = Pooling(pool_h=2, pool_w=2, stride=2)
        
        # 全连接层
        self.layers['Flatten'] = Flatten()
        self.layers['Affine1'] = Affine(self.params['W4'], self.params['B4'])
        self.layers['Relu4'] = ReLU()
        self.layers['Affine2'] = Affine(self.params['W5'], self.params['B5'])
        
        self.lastLayer = SoftmaxWithLoss()

    def predict(self,x, batch_size_predict=None):
        """
        Predict outputs.
        If batch_size_predict is None, run single-pass forward (used in training/gradient).
        If integer specified, run batched forward to save memory during evaluation.
        """
        # Transfer numpy arrays to GPU if needed
        if isinstance(x, cp.ndarray):
            x_gpu = x
        else:
            x_gpu = cp.asarray(x)

        if batch_size_predict is None:
            out = x_gpu
            for layer in self.layers.values():
                out = layer.forward(out)
            return out

        # batched prediction for memory-limited evaluation
        N = int(x_gpu.shape[0])
        outputs = []
        for start in range(0, N, batch_size_predict):
            end = min(start + batch_size_predict, N)
            xb = x_gpu[start:end]
            out = xb
            for layer in self.layers.values():
                out = layer.forward(out)
            outputs.append(out)
        # vstack may be large; collect as list of smaller arrays and concatenate along axis=0
        return cp.concatenate(outputs, axis=0)

    def loss(self,x,t):
        if not isinstance(x, cp.ndarray):
            x = cp.asarray(x)
        if not isinstance(t, cp.ndarray):
            t = cp.asarray(t)
            
        y = self.predict(x, batch_size_predict=None)
        return self.lastLayer.forward(y,t)

    def accuracy(self,x,t, batch_size_predict=128, eval_subset=None):
        """
        accuracy: supports batched prediction and optional subset evaluation to reduce memory.
        eval_subset: if int, evaluate only the first `eval_subset` samples from x/t.
        """
        if eval_subset is not None:
            x = x[:eval_subset]
            t = t[:eval_subset]

        if not isinstance(x, cp.ndarray):
            x = cp.asarray(x)
        if not isinstance(t, cp.ndarray):
            t = cp.asarray(t)
            
        y = self.predict(x, batch_size_predict=batch_size_predict)
        y = cp.argmax(y,axis=1)
        if t.ndim != 1 : t = cp.argmax(t,axis=1)
        accuracy = cp.sum(y==t)/float(x.shape[0])
        return float(accuracy)  # Convert from CuPy to float

    def gradient(self,x,t):
        if not isinstance(x, cp.ndarray):
            x = cp.asarray(x)
        if not isinstance(t, cp.ndarray):
            t = cp.asarray(t)
            
        # 计算梯度（使用单次前向传播确保层缓存匹配完整batch）
        self.loss(x,t)
        dout = 1
        dout = self.lastLayer.backward(dout)
        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)
            
        grads = {}
        # 获取所有层的梯度
        # 卷积层1及其BatchNorm
        grads['W1'] = self.layers['Conv1'].dW
        grads['B1'] = self.layers['Conv1'].db
        grads['bn1_gamma'] = self.layers['BatchNorm1'].dgamma
        grads['bn1_beta'] = self.layers['BatchNorm1'].dbeta
        
        # 卷积层2及其BatchNorm
        grads['W2'] = self.layers['Conv2'].dW
        grads['B2'] = self.layers['Conv2'].db
        grads['bn2_gamma'] = self.layers['BatchNorm2'].dgamma
        grads['bn2_beta'] = self.layers['BatchNorm2'].dbeta
        
        # 卷积层3及其BatchNorm
        grads['W3'] = self.layers['Conv3'].dW
        grads['B3'] = self.layers['Conv3'].db
        grads['bn3_gamma'] = self.layers['BatchNorm3'].dgamma
        grads['bn3_beta'] = self.layers['BatchNorm3'].dbeta
        
        # 全连接层
        grads['W4'] = self.layers['Affine1'].dW
        grads['B4'] = self.layers['Affine1'].db
        grads['W5'] = self.layers['Affine2'].dW
        grads['B5'] = self.layers['Affine2'].db
        
        return grads

In [6]:
from tensorflow.keras.datasets import mnist
if __name__ == '__main__':
    (x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = mnist.load_data()
    # ConvNet 需要 (N, C, H, W) 形状的输入，并转移到GPU
    x_train = cp.asarray((x_train_raw.reshape(x_train_raw.shape[0], 1, 28, 28) / 255.0).astype(cp.float32))
    x_test = cp.asarray((x_test_raw.reshape(x_test_raw.shape[0], 1, 28, 28) / 255.0).astype(cp.float32))
    y_train = cp.asarray(to_one_hot(y_train_raw))
    y_test = cp.asarray(to_one_hot(y_test_raw))
    print(f"\n预处理后:")
    print(f"x_train range: [{float(x_train.min()):.3f}, {float(x_train.max()):.3f}]")
    print(f"x_test range: [{float(x_test.min()):.3f}, {float(x_test.max()):.3f}]")
    print("数据已转移到GPU")


预处理后:
x_train range: [0.000, 1.000]
x_test range: [0.000, 1.000]
数据已转移到GPU


#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;亲爱的读者们，我们已经共同走过了许多，以下是第三章的最后内容。笔者的设备搭载了NVIDIA的移动端5060显卡，训练大约耗时35分钟。所以，把你的电脑设置为永不休眠，去喝一杯茶，或者在大雪天温暖的屋子里休憩一会，然后再来看看我们的模型做出了如何表现吧。这个过程中，不必急躁，也不必害怕。

In [ ]:
if __name__ == '__main__':
    network = ConvNet()
    iters = 15000
    train_size = x_train.shape[0]
    batch_size = 128
    train_loss_list = []  
    train_acc_list = []
    test_acc_list = []  
    best_test_acc = 0
    patience = 20  # 早停耐心值
    no_improve_count = 0

    iter_per_epoch = int(cp.ceil(train_size / batch_size)) 
    optimizer = Adam(lr=0.0005, beta1=0.9, beta2=0.999)  # 降低初始学习率
    # L2 weight decay 
    weight_decay = 0.00005  # 降低weight decay强度

    # When evaluating, use a subset and batched predict to avoid OOM
    eval_subset = 1000
    eval_batch_size = 128  # 降低评估batch size以节省显存

    for i in range(iters):
        batch_mask = cp.random.choice(train_size, batch_size)
        x_batch = x_train[batch_mask]
        y_batch = y_train[batch_mask]
        
        # set training mode for layers that support it
        for layer in network.layers.values():
            if hasattr(layer, 'set_training'):
                layer.set_training(True)

        grad = network.gradient(x_batch, y_batch)
        loss = network.loss(x_batch, y_batch)

        # --- apply L2 weight decay to gradients (additive) ---
        # Do not decay biases or BatchNorm gamma/beta
        if 'weight_decay' in globals() and weight_decay is not None and weight_decay != 0.0:
            for k in list(grad.keys()):
                if k.startswith('W') or k.startswith('w'):
                    grad[k] = grad[k] + weight_decay * network.params[k]

        optimizer.update(network.params, grad)

        # compute a simple gradient norm for diagnostics
        try:
            gsum = 0
            for g in grad.values():
                gsum = gsum + cp.sum(g.astype(cp.float32)**2)
            grad_norm = float(cp.sqrt(gsum))
        except Exception:
            grad_norm = float('nan')

        train_loss_list.append(float(loss))  # Convert CuPy scalar to float
        # 在训练循环中添加学习率衰减
        if i % 2000 == 0 and i > 0:
            optimizer.lr *= 0.95  

        if i % iter_per_epoch == 0:
            # set evaluation mode for layers that support it
            for layer in network.layers.values():
                if hasattr(layer, 'set_training'):
                    layer.set_training(False)

            # evaluate on smaller subset with batched predict
            train_acc = network.accuracy(x_train, y_train, batch_size_predict=eval_batch_size, eval_subset=eval_subset)
            test_acc = network.accuracy(x_test, y_test, batch_size_predict=eval_batch_size, eval_subset=eval_subset)
            train_acc_list.append(float(train_acc))  # Convert CuPy scalar to float
            test_acc_list.append(float(test_acc))    # Convert CuPy scalar to float

            if test_acc > best_test_acc:
                best_test_acc = test_acc
                no_improve_count = 0
            else:
                no_improve_count += 1
                
            if no_improve_count >= patience:
                print(f"Early stopping at iteration {i}")
                print(f"Best test accuracy: {best_test_acc:.4f}")
                break
                
            print(f'iter: {i}, train_acc: {train_acc:.4f}, test_acc: {test_acc:.4f}, loss: {loss:.4f}')
            print(f"Learning rate: {optimizer.lr:.6f}, no_improve_count: {no_improve_count}")
            print(f'Best test accuracy so far: {best_test_acc:.4f}')
            # 在前向传播中添加调试输出
            print("W1 mean:", float(cp.mean(network.params['W1'])), "grad mean:", float(cp.mean(grad['W1'])), "grad_norm:", grad_norm)


iter: 0, train_acc: 0.0930, test_acc: 0.1080, loss: 2.9000
Learning rate: 0.000500, no_improve_count: 0
Best test accuracy so far: 0.1080
W1 mean: 0.004178785718977451 grad mean: 0.003824842938944321 grad_norm: 10.940735816955566
iter: 469, train_acc: 0.9830, test_acc: 0.9770, loss: 0.0199
Learning rate: 0.000500, no_improve_count: 0
Best test accuracy so far: 0.9770
W1 mean: 0.0053804838098585606 grad mean: 0.0004681730727937349 grad_norm: 0.5776388049125671
iter: 469, train_acc: 0.9830, test_acc: 0.9770, loss: 0.0199
Learning rate: 0.000500, no_improve_count: 0
Best test accuracy so far: 0.9770
W1 mean: 0.0053804838098585606 grad mean: 0.0004681730727937349 grad_norm: 0.5776388049125671
iter: 938, train_acc: 0.9890, test_acc: 0.9850, loss: 0.0445
Learning rate: 0.000500, no_improve_count: 0
Best test accuracy so far: 0.9850
W1 mean: 0.009007131680846214 grad mean: 0.001470343125283406 grad_norm: 1.0181812047958374
iter: 938, train_acc: 0.9890, test_acc: 0.9850, loss: 0.0445
Learning 

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;亲爱的读者，只要微笑就可以了。从90％到98%，再从98%到99.5%，我们胜利地完成了伊始的任务。说实话，笔者在撰写此文档时仅仅是初出茅庐的学徒，因此每一格代码对于笔者而言都是学习的过程。如果你喜欢笔者这样从底层书写大模型的方式，那么笔者再高兴不过了。不过在这里要告诉大家一个消息：后续的章节恐怕要放弃这样完全从零开始的构建过程。也就是说，笔者决定使用现有的成熟框架，如Tensorflow或Pytorch。

#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;为做出这个决定，笔者思考了很久。实际上，我们已经基本了解了一个神经网络在训练过程中所需的各种方法。线性层,激活函数,优化器,初始化,正则化,卷积层与池化层等等，这些知识足以支撑我们在未来的旅途中走过非常遥远的距离。与此同时，从零开始构建的弊端在本章集中爆发了：笔者花了非常长的时间处理BatchNorm对卷积层的兼容性问题与预测函数对显存的OOM问题。这样的问题还有许多，笔者只能不厌其烦地debug。这是一个冗长枯燥的过程，完全违背了笔者书写此文档的初衷。于是笔者想：恐怕我们的从零构建的复杂程度已经足以引起质变，所以，到此为止吧。

# &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #f470b0ff;">旅途还没有结束！！</span>

### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #e05c14ff;">一个成熟的模型开发者，必须拥有接受新鲜事物的能力。</span>因此，掌握成熟便利的框架，对于我们而言本就是必不可少的路途。请露出高兴的神色，带着我们新一轮的软件升级，前往下一章吧！: )